# RecipeNLG Dataset Preparation

This CPU-friendly notebook:

1. Mounts Google Drive.
2. Downloads RecipeNLG from Kaggle.
3. Reads the large CSV in chunks.
4. Cleans and reproducibly samples recipes.
5. Removes exact duplicates.
6. Saves fixed train, validation, and test CSV files.

Use a standard CPU runtime; no GPU is needed.


## 1. Install Kaggle

In [ ]:
!pip -q install kaggle

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Create project directories

**Google Drive Configuration**

This notebook stores datasets, model checkpoints, and generated results in Google Drive.

If your project folder is located elsewhere, update `PROJECT_DIR` below.

Expected project structure:

```text
RecipeGPT/
├── data/
├── models/
└── results/
```

In [ ]:
from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# =========================
# Project Configuration
# =========================
# Change this only when your Drive location is different.
PROJECT_DIR = Path("/content/drive/Shareddrives/RecipeGPT")
# Example for personal Drive:
# PROJECT_DIR = Path("/content/drive/MyDrive/RecipeGPT")

DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
# Temporary folder for downloaded files
DOWNLOAD_DIR = Path("/content/recipenlg")

# Create folders if they don't exist
for directory in [DATA_DIR, MODEL_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Temporary download directory: {DOWNLOAD_DIR}")

## 4. Upload your Kaggle API token

Download `kaggle.json` from your Kaggle account settings and upload it below.


In [ ]:
from google.colab import files

uploaded = files.upload()

if "kaggle.json" not in uploaded:
    raise FileNotFoundError("Please upload kaggle.json.")

## 5. Configure Kaggle authentication

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle authentication configured.")

## 6. Download and unzip RecipeNLG

In [ ]:
!kaggle datasets download \
    -d paultimothymooney/recipenlg \
    -p /content/recipenlg \
    --unzip

## 7. Locate and preview the CSV

In [ ]:
import pandas as pd

csv_files = list(DOWNLOAD_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV found in {DOWNLOAD_DIR}.")

FULL_CSV_PATH = csv_files[0]

print("Dataset:", FULL_CSV_PATH)
print("Size (GB):", round(FULL_CSV_PATH.stat().st_size / 1e9, 2))

preview = pd.read_csv(FULL_CSV_PATH, nrows=5)
print("Columns:", preview.columns.tolist())
preview

## 8. Configuration

In [ ]:
SEED = 42

INITIAL_SAMPLE_SIZE = 26_000
FINAL_SAMPLE_SIZE = 24_000

TRAIN_SIZE = 20_000
VALIDATION_SIZE = 2_000
TEST_SIZE = 2_000

CHUNK_SIZE = 50_000

assert TRAIN_SIZE + VALIDATION_SIZE + TEST_SIZE == FINAL_SAMPLE_SIZE

## 9. Cleaning helpers

In [ ]:
import ast
import random
import numpy as np

random.seed(SEED)
np.random.seed(SEED)

def parse_list_text(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""

    text = str(value).strip()
    if not text:
        return ""

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple)):
            return " ".join(
                str(item).strip()
                for item in parsed
                if str(item).strip()
            )
    except (ValueError, SyntaxError):
        pass

    return text

def clean_recipe(row):
    title = str(row.get("title", "")).strip()
    ingredients = parse_list_text(row.get("ingredients", ""))
    directions = parse_list_text(row.get("directions", ""))

    if len(title) < 2 or len(ingredients) < 10 or len(directions) < 20:
        return None

    return {
        "title": title,
        "ingredients": ingredients,
        "directions": directions,
    }

## 10. Reservoir sample the full dataset

This selects a random fixed-size subset without loading the complete CSV into memory.


In [ ]:
reservoir = []
valid_seen = 0

reader = pd.read_csv(
    FULL_CSV_PATH,
    usecols=["title", "ingredients", "directions"],
    chunksize=CHUNK_SIZE,
)

for chunk_number, chunk in enumerate(reader, start=1):
    for row in chunk.to_dict(orient="records"):
        recipe = clean_recipe(row)
        if recipe is None:
            continue

        valid_seen += 1

        if len(reservoir) < INITIAL_SAMPLE_SIZE:
            reservoir.append(recipe)
        else:
            replacement_index = random.randint(0, valid_seen - 1)
            if replacement_index < INITIAL_SAMPLE_SIZE:
                reservoir[replacement_index] = recipe

    print(
        f"Chunk {chunk_number}: "
        f"{valid_seen:,} valid recipes processed; "
        f"{len(reservoir):,} sampled"
    )

sample_df = pd.DataFrame(reservoir)
print("Initial sampled shape:", sample_df.shape)
sample_df.head()

## 11. Remove exact duplicates and select final 24,000

In [ ]:
before = len(sample_df)

sample_df = sample_df.drop_duplicates(
    subset=["title", "ingredients", "directions"]
).reset_index(drop=True)

print("Duplicates removed:", before - len(sample_df))

if len(sample_df) < FINAL_SAMPLE_SIZE:
    raise ValueError(
        f"Only {len(sample_df):,} unique recipes remain. "
        "Increase INITIAL_SAMPLE_SIZE and rerun."
    )

sample_df = sample_df.sample(
    n=FINAL_SAMPLE_SIZE,
    random_state=SEED,
).reset_index(drop=True)

print("Final subset shape:", sample_df.shape)

## 12. Save the complete 24,000-recipe subset

In [ ]:
SUBSET_PATH = DATA_DIR / "RecipeNLG_24k.csv"
sample_df.to_csv(SUBSET_PATH, index=False)

print("Saved:", SUBSET_PATH)

## 13. Create and save fixed splits

In [ ]:
shuffled_df = sample_df.sample(
    frac=1,
    random_state=SEED,
).reset_index(drop=True)

train_df = shuffled_df.iloc[:TRAIN_SIZE].copy()
validation_df = shuffled_df.iloc[
    TRAIN_SIZE:TRAIN_SIZE + VALIDATION_SIZE
].copy()
test_df = shuffled_df.iloc[
    TRAIN_SIZE + VALIDATION_SIZE:
].copy()

TRAIN_PATH = DATA_DIR / "train.csv"
VALIDATION_PATH = DATA_DIR / "validation.csv"
TEST_PATH = DATA_DIR / "test.csv"

train_df.to_csv(TRAIN_PATH, index=False)
validation_df.to_csv(VALIDATION_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)

print("Train:", train_df.shape, TRAIN_PATH)
print("Validation:", validation_df.shape, VALIDATION_PATH)
print("Test:", test_df.shape, TEST_PATH)

## 14. Validate files and split overlap

In [ ]:
def create_key(frame):
    return (
        frame["title"].str.lower().str.strip()
        + "||"
        + frame["ingredients"].str.lower().str.strip()
        + "||"
        + frame["directions"].str.lower().str.strip()
    )

for name, path in {
    "RecipeNLG_24k.csv": SUBSET_PATH,
    "train.csv": TRAIN_PATH,
    "validation.csv": VALIDATION_PATH,
    "test.csv": TEST_PATH,
}.items():
    frame = pd.read_csv(path)
    print(
        name,
        "rows =", len(frame),
        "missing values =", int(frame.isna().sum().sum()),
    )

train_keys = set(create_key(train_df))
validation_keys = set(create_key(validation_df))
test_keys = set(create_key(test_df))

print("Train-validation overlap:", len(train_keys & validation_keys))
print("Train-test overlap:", len(train_keys & test_keys))
print("Validation-test overlap:", len(validation_keys & test_keys))

assert len(train_keys & validation_keys) == 0
assert len(train_keys & test_keys) == 0
assert len(validation_keys & test_keys) == 0

print("Validation passed.")

## 15. Inspect prepared examples

In [ ]:
for split_name, frame in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df),
]:
    example = frame.iloc[0]

    print("\n" + "=" * 80)
    print(split_name.upper())
    print("=" * 80)
    print("Title:", example["title"])
    print("\nIngredients:", example["ingredients"])
    print("\nDirections:", example["directions"][:500])

## Final output

The notebook creates:

```text
MyDrive/recipe_gpt_project/data/
├── RecipeNLG_24k.csv
├── train.csv
├── validation.csv
└── test.csv
```

Use these fixed split files in the GPT-2 training notebook.
